## 1. Математическая модель DEA

Есть 15 школ (DMU).

### Входы:
- $x_{1j}$ - ставки учителей  
- $x_{2j}$ - ставки администрации  
- $x_{3j}$ - число компьютеров  

### Выход:
- $y_{1j}$ - число учеников  

### DEA-модель (CCR, input-oriented)

Для оценки школы $0$ (DMU$_0$) решаем задачу:

$$
\min_{\theta,\lambda_j}\ \theta
$$

при ограничениях

$$
\sum_{j=1}^{15}\lambda_j x_{ij} \le \theta x_{i0}, \quad i=1,2,3
$$

$$
\sum_{j=1}^{15}\lambda_j y_{1j} \ge y_{10}
$$

$$
\lambda_j \ge 0,\quad j = 1,\ldots,15,\qquad \theta \ge 0
$$

Оптимальное значение  
$$
\theta^{*} \in (0,1]
$$  
является оценкой эффективности DMU$_0$.

- Если $\theta^{*} = 1$ (и ограничения выполняются без зазоров), школа считается эффективной.

Подключаем необходимые библиотеки:

In [2]:
import numpy as np
from scipy.optimize import linprog

ModuleNotFoundError: No module named 'scipy'

Данные берем со слайда 38 из файла Bukhvalova-DEA-part 2.pdf:

In [2]:
teachers = np.array([40.2, 18.1, 42.5, 11.0, 24.8,
                     21.1, 13.5, 28.6, 23.5, 15.9,
                     23.2, 26.0, 11.1, 28.8, 19.7])

admin = np.array([2.0, 1.1, 2.1, 0.8, 1.3,
                  1.3, 1.0, 1.3, 1.3, 1.0,
                  1.3, 1.4, 0.8, 1.6, 1.3])

computers = np.array([37, 17, 41, 10, 22,
                      19, 13, 26, 22, 15,
                      22, 25, 11, 26, 18])

students = np.array([602, 269, 648, 188, 420,
                     374, 247, 512, 411, 285,
                     397, 466, 198, 530, 357])

In [3]:
X = np.vstack([teachers, admin, computers])  # 3 x 15 (входы)
Y = np.vstack([students])                    # 1 x 15 (выход)

n_dmu = X.shape[1]

In [1]:
def dea_input_crs(X, Y):
    m, n = X.shape
    s, _ = Y.shape
    efficiencies = []

    for k in range(n):
        c = np.zeros(n + 1)
        c[-1] = 1.0

        A_ub = []
        b_ub = []
        for i in range(m):
            row = np.zeros(n + 1)
            row[:n] = X[i, :]
            row[-1] = -X[i, k]
            A_ub.append(row)
            b_ub.append(0.0)

        for r in range(s):
            row = np.zeros(n + 1)
            row[:n] = -Y[r, :]
            A_ub.append(row)
            b_ub.append(-Y[r, k])

        A_ub = np.array(A_ub)
        b_ub = np.array(b_ub)

        bounds = [(0, None)] * n + [(0, None)]

        res = linprog(c, A_ub=A_ub, b_ub=b_ub,
                      bounds=bounds, method="highs")

        if not res.success:
            efficiencies.append(np.nan)
        else:
            theta = res.x[-1]
            efficiencies.append(theta)

    return np.array(efficiencies)

In [7]:
eff = dea_input_crs(X, Y)

print("Эффективность школ (CCR, input-oriented):")
for i, e in enumerate(eff, start = 1):
    print(f"Школа {i}: эффективность = {e:0.4}")

Эффективность школ (CCR, input-oriented):
Школа 1: эффективность = 0.8267
Школа 2: эффективность = 0.8076
Школа 3: эффективность = 0.8425
Школа 4: эффективность = 0.9287
Школа 5: эффективность = 0.9433
Школа 6: эффективность = 0.9656
Школа 7: эффективность = 0.9942
Школа 8: эффективность = 1.0
Школа 9: эффективность = 0.951
Школа 10: эффективность = 0.974
Школа 11: эффективность = 0.9299
Школа 12: эффективность = 0.9784
Школа 13: эффективность = 0.9693
Школа 14: эффективность = 1.0
Школа 15: эффективность = 0.9847


Интересно, где в этом списке 239 школа?